## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import pandas as pd

from qdrant_client import QdrantClient

In [ ]:
from sem_rag.config.settings import Qdrant
from sem_rag.dataset.metadata_lame import NUMERIC_FIELDS, CATEGORICAL_FIELDS

In [ ]:
client = QdrantClient(url=Qdrant.URL, port=80)

In [ ]:
lame = 1

In [ ]:
if lame:
    numeric_fields = NUMERIC_FIELDS
    collection = Qdrant.COLLECTION_LAME
else:
    numeric_fields = [
        "pixel_size_nm",
        "magnification_x",
        "accelerating_voltage_kv",
        "working_distance_mm",
        "brightness_percent",
        "contrast_percent",
        "scan_speed_setting",
        "frame_average_count",
        "line_average_count",
    ]
    collection = Qdrant.COLLECTION_QWEN

In [ ]:
records = []

offset = None

while True:
    points, offset = client.scroll(
        collection_name=collection,
        offset=offset,
        limit=1000,
        with_payload=True,
        with_vectors=False,
    )

    for point in points:
        payload = point.payload or {}
        payload = payload['meta']

        records.append({
            field: payload.get(field)
            for field in numeric_fields
        })

    if offset is None:
        break

In [ ]:
num_df = pd.DataFrame(records)
num_df

In [ ]:
coverage = pd.DataFrame({
    "total": len(num_df),
    "valid": num_df.notna().sum(),
    "missing": num_df.isna().sum(),
    "coverage_percent": (num_df.notna().mean() * 100).round(2),
})

coverage

In [ ]:
summary = pd.DataFrame({
    "count": num_df[numeric_fields].count(),
    "missing": num_df[numeric_fields].isna().sum(),
    "coverage_%": num_df[numeric_fields].notna().mean() * 100,
    "min": num_df[numeric_fields].min(),
    "max": num_df[numeric_fields].max(),
    "mean": num_df[numeric_fields].mean(),
    "median": num_df[numeric_fields].median(),
    "std": num_df[numeric_fields].std(),
}).round(2)

summary

## Categorical Analysis

In [ ]:
if lame:
    categorical_fields = CATEGORICAL_FIELDS
else:
    categorical_fields = [
        "category",
        "detector",
    ]

In [ ]:
records = []

offset = None

while True:
    points, offset = client.scroll(
        collection_name=collection,
        offset=offset,
        limit=1000,
        with_payload=True,
        with_vectors=False,
    )

    for point in points:
        payload = point.payload or {}
        payload = payload['meta']

        records.append({
            field: payload.get(field)
            for field in categorical_fields
        })

    if offset is None:
        break

In [ ]:
cat_df = pd.DataFrame(records)
cat_df

In [ ]:
coverage = pd.DataFrame({
    "total": len(cat_df),
    "valid": cat_df.notna().sum(),
    "missing": cat_df.isna().sum(),
    "coverage_percent": (cat_df.notna().mean() * 100).round(2),
})

coverage

In [ ]:
category_distributions = {}

for field in categorical_fields:
    category_distributions[field] = (
        cat_df[field]
        .dropna()
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

category_distributions